# How to use CVAT to annotate a dataset

Author: Cait Newport
virtual environment: yolo_training_enviro


<b>There are two examples in this notebook:</b>

A. You have a folder of images and labels in YOLO format and want to upload them to CVAT.

B. You have a folder of videos and want to extract frames from them and upload them to CVAT.


## 1. Uploading pre-annotated images


### 1.1 Open CVAT


In [ ]:
# In the terminal, run the following command:
# cvat

cd /projects/cvat
docker compose up -d # This will start the CVAT server

# Open a web browser and go to http://localhost:8080
# You should see the CVAT login page. 
# log in.


### 1.2 Create a new project in CVAT

* Click on the "Projects" tab in the top left corner of the screen.

* Click on the "Create project" button.

* Enter a name for the project.

* Click on the "Create" button.

* You should now see the project in the list of projects.



### 1.3 Upload the images to the project

#### 1.3.1 Uploading pre-annotated images

Before you can upload the images, you need to get your folders in the correct format. If using <b>YOLO format</b> (YOLO 1.1), you should have a folder with the images and a folder with the matching labels.

You will also need to create a file called `obj.names` that contains the names of the classes in the dataset and a file called `obj.data` that contains the path to the images and labels folders.

raw_data/

├── images/

├── labels/

├── train.txt

└── obj.names

you need to compress the folder into a zip file. 

In [1]:
# For YOLO format, you need to make sure that each image has a corresponding label file. 
# You can do this by running the following code:

from pathlib import Path

# Define directories
IMAGE_DIR = Path("triggerfish-detection/raw_data/images")
LABEL_DIR = Path("triggerfish-detection/raw_data/labels")

# Get all image and label files
image_files = set(f.stem for f in IMAGE_DIR.glob('*.[jp][pn][g]'))  # matches .jpg, .jpeg, .png
label_files = set(f.stem for f in LABEL_DIR.glob('*.txt'))

# Find mismatches
images_without_labels = image_files - label_files
labels_without_images = label_files - image_files

# Print summary
print(f"Total images found: {len(image_files)}")
print(f"Total label files found: {len(label_files)}")
print(f"\nMismatches found:")
print(f"- Images without labels: {len(images_without_labels)}")
print(f"- Labels without images: {len(labels_without_images)}")

# Print detailed mismatches
if images_without_labels:
    print("\nImages missing label files:")
    for img in sorted(images_without_labels):
        print(f"- {img}")

if labels_without_images:
    print("\nLabels missing image files:")
    for label in sorted(labels_without_images):
        print(f"- {label}")

# Print confirmation if everything matches
if not images_without_labels and not labels_without_images:
    print("\n✓ Perfect match! Every image has a corresponding label file.")

Total images found: 0
Total label files found: 0

Mismatches found:
- Images without labels: 0
- Labels without images: 0

✓ Perfect match! Every image has a corresponding label file.


#### 1.3.2 Zip Training images

You need to ZIP training images. This can take a few seconds if you have a lot of images.

In [29]:
import os
from pathlib import Path

# Set the name of the zip file
zip_folder_name = "triggerfish_detection/raw_data/upload_images.zip"

# First, let's check if train.txt exists and create it if needed
images_dir = Path("triggerfish_detection/raw_data/images")

# Create zip file of all images in the images folder
print("\nCreating zip file...")
!zip -v {zip_folder_name} triggerfish_detection/raw_data/images/*.jpg

# Verify zip was created
zip_path = Path("triggerfish_detection/raw_data/images.zip")
if zip_path.exists():
    print(f"\nZip file created successfully at {zip_path}")
    print(f"Size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("\nError: Zip file was not created")


Creating zip file...
  adding: triggerfish_detection/raw_data/images/image_0000.jpg 	(in=130699) (out=128118) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0001.jpg 	(in=127656) (out=125275) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0002.jpg 	(in=132042) (out=129429) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0003.jpg 	(in=127749) (out=125328) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0004.jpg 	(in=127103) (out=124352) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0005.jpg 	(in=128645) (out=126083) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0006.jpg 	(in=128012) (out=125311) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0007.jpg 	(in=122064) (out=119429) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/image_0008.jpg 	(in=126243) (out=123760) (deflated 2%)
  adding: triggerfish_detection/raw_data/images/ima

#### 3.3 Create files needed to upload annotations

##### 3.3.1 Create a folder that will be uploaded to CVAT

The next step is to create a folder with the correct structure and which contains the labels and some associated files in the correct format.


In [30]:
# Create a folder called data in the raw_data folder
data_dir = Path("triggerfish_detection/raw_data/data")
data_dir.mkdir(parents=True, exist_ok=True)


##### 3.3.2 Create a folder called obj_train_data in the data folder

The labels will be in this folder.

In [31]:

# Create a folder called obj_train_data in the data folder
obj_train_data_dir = data_dir / "obj_train_data"
obj_train_data_dir.mkdir(parents=True, exist_ok=True)

# Copy the labels from raw_data/labels to data/obj_train_data
# Copy only .txt label files
source_dir = Path("triggerfish_detection/raw_data/labels")
for file in source_dir.glob("*.txt"):
    destination = obj_train_data_dir / file.name
    with open(file, 'rb') as src, open(destination, 'wb') as dst:
        dst.write(src.read())

# check how many files are in the obj_train_data folder
print(f"Number of files in obj_train_data: {len(list(obj_train_data_dir.glob('*')))}")

Number of files in obj_train_data: 8787


##### 3.3.3 Create a file called obj.names

The obj.names file should contain the names of the classes in the dataset. For example:

triggerfish

In [32]:
# Create obj.names file
obj_names_path = Path("triggerfish_detection/raw_data/data/obj.names")
label_categories = ['triggerfish']

# Write 'triggerfish' to the file
with open(obj_names_path, 'w') as f:
    for category in label_categories:
        f.write(f"{category}\n")

# Verify the file was created
if obj_names_path.exists():
    print(f"Created {obj_names_path} successfully")
    with open(obj_names_path, 'r') as f:
        print("\nContents:")
        print(f.read())
else:
    print("Error: File was not created")

Created triggerfish_detection/raw_data/data/obj.names successfully

Contents:
triggerfish



##### 3.3.4 Create a file called obj.data

The obj.data file should contain the following:

classes = 1

train = data/train.txt

names = data/obj.names

backup = backup/


In [33]:
# Create obj.data file
obj_data_path = Path("triggerfish_detection/raw_data/data/obj.data")

# Count the number of classes in the dataset
num_classes = len(list(label_categories))
print(f"Number of classes: {num_classes}")

# Write the configuration to the file
content = f"""classes = {num_classes}
train = data/train.txt
names = data/obj.names
backup = backup/"""

with open(obj_data_path, 'w') as f:
    f.write(content)

# Verify the file was created
if obj_data_path.exists():
    print(f"Created {obj_data_path} successfully")
    print("\nContents:")
    with open(obj_data_path, 'r') as f:
        print(f.read())
else:
    print("Error: File was not created")

Number of classes: 1
Created triggerfish_detection/raw_data/data/obj.data successfully

Contents:
classes = 1
train = data/train.txt
names = data/obj.names
backup = backup/


##### 3.3.5 Create a file called train.txt

The train.txt file should contain the path to each image in the dataset. For example:

data/obj_train_data/image1.jpg

data/obj_train_data/image2.jpg

data/obj_train_data/image3.jpg


In [34]:
from pathlib import Path

# Define paths
images_dir = Path("triggerfish_detection/raw_data/images")
data_dir = Path("triggerfish_detection/raw_data/data")
train_txt_path = data_dir / "train.txt"  # This puts train.txt in the data directory

# Ensure data directory exists
data_dir.mkdir(parents=True, exist_ok=True)

# Create train.txt with list of image paths
with open(train_txt_path, 'w') as f:
    for img_path in sorted(images_dir.glob('*.[jp][pn][g]')):  # matches .jpg, .jpeg, .png
        # Write path in the format: data/obj_train_data/filename.jpg
        relative_path = f"data/obj_train_data/{img_path.name}"
        f.write(relative_path + '\n')

# Verify the file was created and show first few entries
if train_txt_path.exists():
    print(f"Created {train_txt_path} successfully")
    print("\nFirst 5 entries:")
    with open(train_txt_path, 'r') as f:
        for i, line in enumerate(f):
            if i < 5:  # Show first 5 lines
                print(line.strip())
            else:
                break
    print("...")
else:
    print("Error: File was not created")

Created triggerfish_detection/raw_data/data/train.txt successfully

First 5 entries:
data/obj_train_data/image_0000.jpg
data/obj_train_data/image_0001.jpg
data/obj_train_data/image_0002.jpg
data/obj_train_data/image_0003.jpg
data/obj_train_data/image_0004.jpg
...


##### 3.3.6 ZIP the data folder

On a MAC, DO NOT zip the folder from the Finder as you will get an error. Also, if you just use the zip command, you will get an error. Instead, use the following command in the terminal:

In [39]:
# Name the zip folder
zip_folder_name = "triggerfish_detection/raw_data/upload_labels.zip"

# zip the data folder
!tar -cvf {zip_folder_name} -C triggerfish_detection/raw_data/data .

# check the size of the zip file
!ls -lh {zip_folder_name}

a .
a ./obj.data
a ./obj_train_data
a ./train.txt
a ./obj.names
a ./obj_train_data/image_2337.txt
a ./obj_train_data/image_5458.txt
a ./obj_train_data/image_4746.txt
a ./obj_train_data/image_3029.txt
a ./obj_train_data/image_0520.txt
a ./obj_train_data/image_6151.txt
a ./obj_train_data/image_1158.txt
a ./obj_train_data/image_6637.txt
a ./obj_train_data/image_7529.txt
a ./obj_train_data/image_0246.txt
a ./obj_train_data/image_4020.txt
a ./obj_train_data/image_3997.txt
a ./obj_train_data/image_2451.txt
a ./obj_train_data/image_4034.txt
a ./obj_train_data/image_3983.txt
a ./obj_train_data/image_2445.txt
a ./obj_train_data/image_6623.txt
a ./obj_train_data/image_0252.txt
a ./obj_train_data/image_0534.txt
a ./obj_train_data/image_8168.txt
a ./obj_train_data/image_6145.txt
a ./obj_train_data/image_2323.txt
a ./obj_train_data/image_4752.txt
a ./obj_train_data/image_7273.txt
a ./obj_train_data/image_8140.txt
a ./obj_train_data/image_1602.txt
a ./obj_train_data/image_5464.txt
a ./obj_train_data

## 3.3 Create a CVAT task with your labels:

* Click on the "Tasks" tab in the top left corner of the screen.

* Click on the "Create task" button.

* Enter a name for the task.

* drag your zip file into the "Upload" section.

* Click on the "Submit & Continue" button.

* You should now see the task in the list of tasks.


Before you upload the images, you need to compress the folder into a zip file. 

* Click on the "Actions" button in the top right corner of the screen.

* Click on the "Import dataset" button.

* Select the images you want to upload.

* Click on the "Upload" button.

* You should now see the images in the project.



### 3. Create a new task

### 4. Annotate the images

### 5. Download the annotations




## B. Uploading new frames to be annotated

1. Extract frames from videos
2. Upload frames to CVAT
3. Annotate frames
4. Download annotations
5. Use annotations to train model
6. Use model to annotate new videos


### 1. Extract frames from videos

In [27]:
# Clear out the old version from Python's cache
import importlib
import extract_frames_v2    
importlib.reload(extract_frames_v2)

# Now import the process_videos function from the reloaded module
from extract_frames_v2 import process_videos
import os

# Set your parameters
top_folder = "/Volumes/RFS/Triggerfish Navigation/originaldata/"
output_folder = "/Users/user/projects/VideoUtilities/triggerfish_detection/fieldwork_2024/images/"
num_folders = 15  # Number of random folders to process, essentially the number of videos to process
frames_per_video = 20

print(f"Processing videos from {num_folders} folders from {top_folder} to {output_folder}")

# Debug: Check top_folder exists
if not os.path.exists(top_folder):
    print(f"Error: {top_folder} does not exist")
    exit(1)

# Debug: Check output_folder exists
if not os.path.exists(output_folder):
    print(f"{output_folder} does not exist")
    os.makedirs(output_folder)
    print(f"Created {output_folder}")
    exit(1)

# Process the videos
csv_path = process_videos(
    top_folder=top_folder,
    output_folder=output_folder,
    num_folders=num_folders,
    frames_per_video=frames_per_video,
    csv_filename="frame_list.csv"
)

if csv_path:
    print(f"Processing complete! CSV file saved at: {csv_path}")
else:
    print("Processing failed!")

Processing videos from 15 folders from /Volumes/RFS/Triggerfish Navigation/originaldata/ to /Users/user/projects/VideoUtilities/triggerfish_detection/fieldwork_2024/images/
/Users/user/projects/VideoUtilities/triggerfish_detection/fieldwork_2024/images/ does not exist
Created /Users/user/projects/VideoUtilities/triggerfish_detection/fieldwork_2024/images/

Searching for folders in: /Volumes/RFS/Triggerfish Navigation/originaldata/

Total valid folders found: 314
Valid folders: ['/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_26/Cait/Left', '/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_26/Cait/Right', '/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_26/Valerio/Left', '/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_26/Valerio/Right', '/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_27/AM1/Cait/Left', '/Volumes/RFS/Triggerfish Navigation/originaldata/Lizard/2024_02_27/AM1/Cait/Right', '/Volumes/RFS/Trigge

### 2. Upload frames to CVAT
Select the images from the folder and upload them to CVAT. You don't need to zip a folder first. 

### 3. Annotate frames

### 4. Download annotations

### 5. Use annotations to train model




## 3 Extracting frames from videos listed in a text file

In [15]:

import cv2
import pandas as pd
from pathlib import Path
import os

def create_frame_name(video_path, frame_idx):
    """Create frame name from video path and frame index."""
    # Convert path to Path object
    path = Path(video_path)
    
    # Get path relative to the base directory
    try:
        relative_path = path.relative_to('/Volumes/RFS/Triggerfish Navigation/originaldata')
    except ValueError:
        # If the path doesn't start with the expected base, use the full path
        relative_path = path
        
    # Convert path parts to list and join with underscores
    name_parts = list(relative_path.parts)
    # Remove .MP4 from the last part
    name_parts[-1] = name_parts[-1].replace('.MP4', '')
    name_base = '_'.join(name_parts)
    name_base = name_base.replace(' ', '_')
    
    # Add frame index
    return f"{name_base}_frame{frame_idx:04d}.jpg"

def extract_frames(input_file, output_dir):
    """Extract frames from videos based on input file specifications."""
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Define the base path that needs to be prepended
    BASE_PREFIX = '/Volumes/RFS/Triggerfish '
    
    # Read the input file and clean up paths
    df = pd.read_csv(input_file, sep='\s+')
    
    # Process each video
    for _, row in df.iterrows():
        # Get path components directly
        base_path = row['file_path']
        file_name = row['file_name']
        start_frame = int(row['start_frame'])
        num_frames = int(row['num_frame'])
        skip_by = int(row['skip_by'])  # New parameter
        
        # Combine path with video name, ensuring the full path prefix is included
        video_path = f"{BASE_PREFIX}{base_path}/{file_name}"
        
        # Debug print
        print(f"\nProcessing video: {video_path}")
        print(f"Requested start frame: {start_frame}")
        print(f"Number of frames to extract: {num_frames}")
        print(f"Skipping every {skip_by} frames")

        # Check if file exists
        if not os.path.exists(video_path):
            print(f"Error: File does not exist at {video_path}")
            continue
            
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: Could not open video {video_path}")
            continue
            
        # Get total frames in video
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"Total frames in video: {total_frames}")
        
        frames_extracted = 0
        current_frame = start_frame
        
        while frames_extracted < num_frames:
            # Set position to current frame
            cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
            
            ret, frame = cap.read()
            if not ret:
                print(f"Error: Could not read frame {current_frame} from {video_path}")
                break
                
            frame_name = create_frame_name(video_path, current_frame)
            output_path = os.path.join(output_dir, frame_name)
            
            # Save frame
            success = cv2.imwrite(output_path, frame)
            if success:
                print(f"Saved frame {current_frame} to {output_path}")
                frames_extracted += 1
            else:
                print(f"Failed to save frame {current_frame}")
            
            # Skip ahead by skip_by frames
            current_frame += skip_by
            
            # Check if we've reached the end of the video
            if current_frame >= total_frames:
                print(f"Reached end of video after extracting {frames_extracted} frames")
                break
        
        cap.release()
        
        print(f"Processed {video_path}: extracted {num_frames} frames starting from frame {start_frame}")

# Example usage
input_file = '/Users/user/projects/VideoUtilities/triggerfish_detection/datasets/videos_for_frame_extract_v2.txt'  # Path to your input file
output_dir = '/Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images'  # Directory where frames will be saved

# Extract frames
extract_frames(input_file, output_dir)


Processing video: /Volumes/RFS/Triggerfish Navigation/originaldata/Maldives/2024_02_15/PM3/Cait/Left/GX020010.MP4
Requested start frame: 1180
Number of frames to extract: 50
Skipping every 5 frames
Total frames in video: 153600
Saved frame 1180 to /Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images/Maldives_2024_02_15_PM3_Cait_Left_GX020010_frame1180.jpg
Saved frame 1185 to /Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images/Maldives_2024_02_15_PM3_Cait_Left_GX020010_frame1185.jpg
Saved frame 1190 to /Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images/Maldives_2024_02_15_PM3_Cait_Left_GX020010_frame1190.jpg
Saved frame 1195 to /Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images/Maldives_2024_02_15_PM3_Cait_Left_GX020010_frame1195.jpg
Saved frame 1200 to /Users/user/projects/VideoUtilities/triggerfish_detection/datasets/dataset_v5/images/Maldives_2024_02_1

In [16]:
# Check if there are any image names that match in dataset_v4/images and dataset_v5/images

from pathlib import Path

# Define directories
DIR_V4 = Path("./triggerfish_detection/datasets/dataset_v4/images")
DIR_V5 = Path("./triggerfish_detection/datasets/dataset_v5/images")

# Get all image filenames from both directories
images_v4 = set(f.name for f in DIR_V4.glob('*.[jp][pn][g]'))  # matches .jpg, .jpeg, .png
images_v5 = set(f.name for f in DIR_V5.glob('*.[jp][pn][g]'))

# Find matching images
matching_images = images_v4.intersection(images_v5)

# Print summary
print(f"Dataset v4 images: {len(images_v4)}")
print(f"Dataset v5 images: {len(images_v5)}")
print(f"Matching images: {len(matching_images)}")

# Print matching images if any exist
if matching_images:
    print("\nMatching image names:")
    for img in sorted(matching_images):
        print(f"- {img}")
else:
    print("\n✓ No matching image names found between the datasets!")

Dataset v4 images: 698
Dataset v5 images: 450
Matching images: 0

✓ No matching image names found between the datasets!


### Check for unpaired image and label files

In [23]:
from pathlib import Path

# Define directories
IMAGE_DIR = Path("./triggerfish_detection/datasets/dataset_v6/images")
LABEL_DIR = Path("./triggerfish_detection/datasets/dataset_v6/labels")

# Get all image and label files
image_files = set(f.stem for f in IMAGE_DIR.glob('*.[jp][pn][g]'))  # matches .jpg, .jpeg, .png
label_files = set(f.stem for f in LABEL_DIR.glob('*.txt'))

# Find mismatches
images_without_labels = image_files - label_files
labels_without_images = label_files - image_files

# Print summary
print(f"Total images found: {len(image_files)}")
print(f"Total label files found: {len(label_files)}")
print(f"\nMismatches found:")
print(f"Images without labels: {len(images_without_labels)}")
print(f"Labels without images {len(labels_without_images)} ")

# Print detailed mismatches
if images_without_labels:
    print("\nImages missing label files:")
    for img in sorted(images_without_labels):
        print(f"- {img}")
else:
    print("\n✓ Perfect match! Every image has a corresponding label file.")

if labels_without_images:
    print("\nLabels missing image files:")
    for label in sorted(labels_without_images):
        print(f"-{label}")
else:
    print("\n✓ Perfect match! Every label has a corresponding image file")

Total images found: 0
Total label files found: 0

Mismatches found:
Images without labels: 0
Labels without images 0 

✓ Perfect match! Every image has a corresponding label file.

✓ Perfect match! Every label has a corresponding image file


In [27]:
# Count number of labels in a folder have no annotation

from pathlib import Path

# Define directory
LABEL_DIR = Path("./triggerfish_detection/datasets/dataset_9000/labels")

empty_files = []

# Check each label file
for label_file in LABEL_DIR.glob('*.txt'):
    with open(label_file, 'r') as f:
        content = f.read().strip()
        
        # Check if file is empty
        if not content:
            empty_files.append(label_file.name)
            continue   
        
# Calculate totals and percentages
total_files = len(list(LABEL_DIR.glob('*.txt')))
empty_percent = (len(empty_files) / total_files) * 100

# Print results
print(f"Total label files checked: {total_files}")
print(f"Empty files: {len(empty_files)} ({empty_percent:.1f}%)")

# Print details of empty files
if empty_files:
    print("\nEmpty label files:")
    for file in sorted(empty_files):
        print(f"- {file}")

Total label files checked: 10102
Empty files: 796 (7.9%)

Empty label files:
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18100.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18105.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18110.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18115.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18120.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18125.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18130.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18135.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18140.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18145.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18150.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18155.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18160.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18165.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18170.txt
- Lizard_2024_02_27_AM1_Cait_Left_GX010170_frame18175

## Using ffplay
This is a great tool to look for specific frames in a video. It is part of ffmepg.

This is to be done in the TERMINAL.

Controls:

<b>Space:</b> Play/pause

<b>S:</b> Step forward one frame when paused

<b>Left/Right arrows:</b> Seek by 10 seconds

<b>Up/Down arrows:</b> Seek by 1 minute



In [ ]:
# For the TERMINAL

cd projects/VideoUtilities
ffmpeg -version # Check you have it installed

ffplay -vf "drawtext=text='%{n}':x=10:y=10:fontsize=24:fontcolor=white" "/Volumes/RFS/Triggerfish Navigation/originaldata/Maldives/2024_02_15/PM3/Cait/Left/GX010010.MP4"
# A screen will pop up with the video

In [22]:
# Copy images from triggerfish_detection/raw_data that have a corresponding .txt file in triggerfish_detection/cleaned data.
# Move them into triggerfish_detection/datasets/dataset_9000/images

from pathlib import Path
import shutil

# Define directories
SOURCE_IMG_DIR = Path("./triggerfish_detection/raw_data/images")
LABEL_DIR = Path("./triggerfish_detection/cleaned_dataset/obj_train_data")
DEST_DIR = Path("./triggerfish_detection/datasets/dataset_9000/images")

# Create destination directory if it doesn't exist
DEST_DIR.mkdir(parents=True, exist_ok=True)

# Get all label files
label_files = set(f.stem for f in LABEL_DIR.glob('*.txt'))

# Counter for copied files
copied_count = 0

# Copy images that have matching label files
for img_path in SOURCE_IMG_DIR.glob('*.[jp][pn][g]'):  # matches .jpg, .jpeg, .png
    if img_path.stem in label_files:
        dest_path = DEST_DIR / img_path.name
        shutil.copy2(img_path, dest_path)
        copied_count += 1
        print(f"Copied: {img_path.name}")

# Print summary
print(f"\nSummary:")
print(f"Total label files found: {len(label_files)}")
print(f"Total images copied: {copied_count}")

# Verify the copy
dest_files = list(DEST_DIR.glob('*.[jp][pn][g]'))
print(f"Files in destination directory: {len(dest_files)}")

# Debug: Print some paths to verify they're correct
print("\nDebug - Checking paths:")
print(f"Source image directory exists: {SOURCE_IMG_DIR.exists()}")
print(f"Label directory exists: {LABEL_DIR.exists()}")
print(f"Destination directory exists: {DEST_DIR.exists()}")


Copied: image_8240.jpg
Copied: image_7173.jpg
Copied: image_1502.jpg
Copied: image_5764.jpg
Copied: image_3315.jpg
Copied: image_3473.jpg
Copied: image_5002.jpg
Copied: image_1264.jpg
Copied: image_8526.jpg
Copied: image_7615.jpg
Copied: image_1270.jpg
Copied: image_8532.jpg
Copied: image_7601.jpg
Copied: image_3467.jpg
Copied: image_4308.jpg
Copied: image_5016.jpg
Copied: image_2779.jpg
Copied: image_5770.jpg
Copied: image_3301.jpg
Copied: image_0608.jpg
Copied: image_8254.jpg
Copied: image_7167.jpg
Copied: image_6279.jpg
Copied: image_1516.jpg
Copied: image_2037.jpg
Copied: image_5758.jpg
Copied: image_4446.jpg
Copied: image_3329.jpg
Copied: image_5980.jpg
Copied: image_0620.jpg
Copied: image_6251.jpg
Copied: image_1258.jpg
Copied: image_6537.jpg
Copied: image_7629.jpg
Copied: image_0146.jpg
Copied: image_2989.jpg
Copied: image_4320.jpg
Copied: image_2751.jpg
Copied: image_4334.jpg
Copied: image_2745.jpg
Copied: image_6523.jpg
Copied: image_0152.jpg
Copied: image_0634.jpg
Copied: ima